In [3]:
import pandas as pd
import numpy as np
import openpyxl
from decimal import Decimal, ROUND_HALF_UP

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


def redondear_nom(v, decimales=0):
    """
    Redondeo aritmético estándar (Round Half Up) requerido por
    normativas de calidad del aire.
    """
    if pd.isna(v) or v is None:
        return np.nan
    try:
        d = Decimal(str(float(v)))
        if decimales == 0:
            return int(d.quantize(Decimal('1'), rounding=ROUND_HALF_UP))
        else:
            fmt = '0.' + '0' * decimales
            return float(d.quantize(Decimal(fmt), rounding=ROUND_HALF_UP))
    except Exception:
        return np.nan


def calcular_nowcast_serie(series):
    """
    Calcula el indicador NowCast hora a hora para partículas (PM10 / PM2.5)
    basado en la ventana ponderada de 12 horas.
    """
    nowcasts = []
    for i in range(len(series)):
        ventana = series.iloc[max(0, i - 11):i + 1]
        validos = ventana.dropna()
        if len(validos) < 2:
            nowcasts.append(np.nan)
            continue
        c_max = validos.max()
        c_min = validos.min()
        w = 1 - (c_max - c_min) / c_max if c_max > 0 else 1.0
        if w < 0.5:
            w = 0.5

        pesos = [w**(len(validos) - 1 - idx) for idx in range(len(validos))]
        val_nowcast = (validos * pesos).sum() / sum(pesos)
        nowcasts.append(val_nowcast)
    return pd.Series(nowcasts, index=series.index)


def procesar_calidad_aire_diaria_nom2023(
    input_file="BDPIN_Marzo_2024.xlsx",
    output_file="Calculos_Diarios_Aire_y_Salud.xlsx",
    sheet_name="Data"
):
    print(f"Cargando archivo: {input_file} (Hoja: '{sheet_name}')...")

    df = pd.read_excel(input_file, sheet_name=sheet_name)
    contaminantes = ['PM2.5', 'PM10', 'O3']

    # 1. Validar columnas necesarias
    columnas_requeridas = ['DATE'] + contaminantes
    faltantes = [c for c in columnas_requeridas if c not in df.columns]

    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en el archivo: {faltantes}")

    # 2. Convertir fechas y asegurar orden cronológico
    df['DATETIME'] = pd.to_datetime(df['DATE'], errors='coerce')
    if df['DATETIME'].isna().all():
        raise ValueError("No se pudieron interpretar las fechas de la columna DATE.")

    df['DIA'] = df['DATETIME'].dt.date
    df = df.sort_values('DATETIME').reset_index(drop=True)

    for c in contaminantes:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Convertir O3 de ppb a ppm si viniera en ppb
    o3_validos = df['O3'].dropna()
    if not o3_validos.empty and o3_validos.median() > 1.0:
        print("Nota: Se detectó O3 en ppb. Convirtiendo a ppm...")
        df['O3'] = df['O3'] / 1000.0

    # 3. Pre-cálculo de Series Móviles (O3 8h y Nowcast)
    # O3: Promedio móvil 8h
    o3_8h_list = []
    for i in range(len(df)):
        v_o3 = df['O3'].iloc[max(0, i-7):i+1].dropna()
        o3_8h_list.append(v_o3.mean() if len(v_o3) >= 6 else np.nan)
    df['O3_8H'] = o3_8h_list

    # PM Nowcast
    df['PM10_NOWCAST'] = calcular_nowcast_serie(df['PM10'])
    df['PM2.5_NOWCAST'] = calcular_nowcast_serie(df['PM2.5'])

    # 4. Funciones de Categorización (NOM-172-SEMARNAT-2023)
    def cat_pm25_2023(v):
        if pd.isna(v): return np.nan
        if v <= 15.0: return 'Buena'
        elif v <= 41.0: return 'Aceptable'
        elif v <= 79.0: return 'Mala'
        elif v <= 130.0: return 'Muy Mala'
        else: return 'Extremadamente Mala'

    def cat_pm10_2023(v):
        if pd.isna(v): return np.nan
        if v <= 45.0: return 'Buena'
        elif v <= 70.0: return 'Aceptable'
        elif v <= 132.0: return 'Mala'
        elif v <= 213.0: return 'Muy Mala'
        else: return 'Extremadamente Mala'

    def cat_o3(v):
        if pd.isna(v): return np.nan
        if v <= 0.058: return 'Buena'
        elif v <= 0.090: return 'Aceptable'
        elif v <= 0.135: return 'Mala'
        elif v <= 0.175: return 'Muy Mala'
        else: return 'Extremadamente Mala'

    severity_map = {'Buena': 1, 'Aceptable': 2, 'Mala': 3, 'Muy Mala': 4, 'Extremadamente Mala': 5}
    inv_severity_map = {1: 'Buena', 2: 'Aceptable', 3: 'Mala', 4: 'Muy Mala', 5: 'Extremadamente Mala'}

    # 5. Agregación Diaria
    resultados = []
    for dia, group in df.groupby('DIA'):
        row_res = {'DATE': dia}
        cats_dia = {}

        # --- PM2.5 ---
        s_pm25 = group['PM2.5'].dropna()
        v_pm25 = len(s_pm25)
        cumple_pm25 = v_pm25 >= 18
        row_res['Validos_PM2.5'] = v_pm25
        row_res['Suficiencia_PM2.5'] = 'Cumple' if cumple_pm25 else 'No cumple'

        row_res['PM2.5_AVG_24H'] = redondear_nom(s_pm25.mean(), 0) if cumple_pm25 else np.nan
        row_res['PM2.5_MAX_1H'] = redondear_nom(s_pm25.max(), 0) if v_pm25 > 0 else np.nan
        nc_pm25 = group['PM2.5_NOWCAST'].dropna()
        row_res['PM2.5_NOWCAST_MAX'] = redondear_nom(nc_pm25.max(), 0) if len(nc_pm25) > 0 else np.nan

        # Categoría basada en promedio de 24h según NOM-172
        cat_pm25 = cat_pm25_2023(row_res['PM2.5_AVG_24H'])
        row_res['Cat_PM2.5'] = cat_pm25
        if pd.notna(cat_pm25): cats_dia['PM2.5'] = cat_pm25

        # --- PM10 ---
        s_pm10 = group['PM10'].dropna()
        v_pm10 = len(s_pm10)
        cumple_pm10 = v_pm10 >= 18
        row_res['Validos_PM10'] = v_pm10
        row_res['Suficiencia_PM10'] = 'Cumple' if cumple_pm10 else 'No cumple'

        row_res['PM10_AVG_24H'] = redondear_nom(s_pm10.mean(), 0) if cumple_pm10 else np.nan
        row_res['PM10_MAX_1H'] = redondear_nom(s_pm10.max(), 0) if v_pm10 > 0 else np.nan
        nc_pm10 = group['PM10_NOWCAST'].dropna()
        row_res['PM10_NOWCAST_MAX'] = redondear_nom(nc_pm10.max(), 0) if len(nc_pm10) > 0 else np.nan

        cat_pm10 = cat_pm10_2023(row_res['PM10_AVG_24H'])
        row_res['Cat_PM10'] = cat_pm10
        if pd.notna(cat_pm10): cats_dia['PM10'] = cat_pm10

        # --- O3 ---
        s_o3 = group['O3'].dropna()
        v_o3 = len(s_o3)
        cumple_o3 = v_o3 >= 18
        row_res['Validos_O3'] = v_o3
        row_res['Suficiencia_O3'] = 'Cumple' if cumple_o3 else 'No cumple'

        row_res['O3_AVG_24H'] = redondear_nom(s_o3.mean(), 3) if cumple_o3 else np.nan
        row_res['O3_MAX_1H'] = redondear_nom(s_o3.max(), 3) if v_o3 > 0 else np.nan
        o3_8h_s = group['O3_8H'].dropna()
        row_res['O3_MAX_8H'] = redondear_nom(o3_8h_s.max(), 3) if len(o3_8h_s) > 0 else np.nan

        # Categoría basada en máximo horario según NOM-172
        cat_o3_val = cat_o3(row_res['O3_MAX_1H'])
        row_res['Cat_O3'] = cat_o3_val
        if pd.notna(cat_o3_val): cats_dia['O3'] = cat_o3_val

        # --- CÁLCULO DE CATEGORÍA GLOBAL ---
        if cats_dia:
            max_sev = max(severity_map[c] for c in cats_dia.values())
            row_res['Cat_Global_Estacion'] = inv_severity_map[max_sev]
            det = [c for c, cat in cats_dia.items() if severity_map[cat] == max_sev]
            row_res['Contaminante_Determinante'] = ', '.join(det)
        else:
            row_res['Cat_Global_Estacion'] = np.nan
            row_res['Contaminante_Determinante'] = 'N/A'

        resultados.append(row_res)

    # 6. Generar DataFrame Final con el orden de columnas exacto
    df_res = pd.DataFrame(resultados)

    columnas_finales = [
        'DATE',
        'Validos_PM2.5', 'Suficiencia_PM2.5', 'PM2.5_AVG_24H', 'PM2.5_MAX_1H', 'PM2.5_NOWCAST_MAX', 'Cat_PM2.5',
        'Validos_PM10', 'Suficiencia_PM10', 'PM10_AVG_24H', 'PM10_MAX_1H', 'PM10_NOWCAST_MAX', 'Cat_PM10',
        'Validos_O3', 'Suficiencia_O3', 'O3_AVG_24H', 'O3_MAX_1H', 'O3_MAX_8H', 'Cat_O3',
        'Cat_Global_Estacion', 'Contaminante_Determinante'
    ]
    df_res = df_res[columnas_finales]

    # 7. Generar Excel y aplicar Estilos
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_res.to_excel(writer, sheet_name='Calculos_Diarios', index=False)

    wb = openpyxl.load_workbook(output_file)
    ws = wb['Calculos_Diarios']

    font_header = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
    fill_header = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')
    font_data = Font(name='Segoe UI', size=10)

    thin_border = Border(
        left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'),
        top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9')
    )

    category_colors = {
        'Buena': {'fill': 'C6EFCE', 'font': '006100'},
        'Aceptable': {'fill': 'FFEB9C', 'font': '9C6500'},
        'Mala': {'fill': 'FCE4D6', 'font': 'C65911'},
        'Muy Mala': {'fill': 'FFC7CE', 'font': '9C0006'},
        'Extremadamente Mala': {'fill': 'E1D5E7', 'font': '622383'}
    }

    suficiencia_colors = {
        'Cumple': {'fill': 'E2EFDA', 'font': '375623'},
        'No cumple': {'fill': 'FCE4D6', 'font': 'C65911'}
    }

    ws.freeze_panes = 'B2'
    ws.row_dimensions[1].height = 32

    # Formato a Encabezados
    for col_num in range(1, ws.max_column + 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = font_header
        cell.fill = fill_header
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    # Formato a Filas de Datos
    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        for cell in row:
            cell.font = font_data
            cell.border = thin_border
            encabezado = str(ws.cell(row=1, column=cell.column).value)
            valor = str(cell.value) if cell.value is not None else ''

            if encabezado == 'DATE':
                cell.number_format = 'yyyy-mm-dd'
                cell.alignment = Alignment(horizontal='center')

            elif encabezado.startswith('O3_'):
                cell.alignment = Alignment(horizontal='right')
                cell.number_format = '0.000'

            elif encabezado.startswith('PM10_') or encabezado.startswith('PM2.5_'):
                cell.alignment = Alignment(horizontal='right')
                cell.number_format = '0'

            elif encabezado.startswith('Suficiencia_'):
                cell.alignment = Alignment(horizontal='center')
                if valor in suficiencia_colors:
                    cell.fill = PatternFill(start_color=suficiencia_colors[valor]['fill'], end_color=suficiencia_colors[valor]['fill'], fill_type='solid')
                    cell.font = Font(name='Segoe UI', size=10, bold=True, color=suficiencia_colors[valor]['font'])

            elif encabezado.startswith('Cat_') or encabezado == 'Contaminante_Determinante':
                cell.alignment = Alignment(horizontal='center')
                if valor in category_colors:
                    cell.fill = PatternFill(start_color=category_colors[valor]['fill'], end_color=category_colors[valor]['fill'], fill_type='solid')
                    cell.font = Font(name='Segoe UI', size=10, bold=True, color=category_colors[valor]['font'])

    # Ajuste automático del ancho de las columnas
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        letra = get_column_letter(col[0].column)
        ws.column_dimensions[letra].width = min(max(max_len + 4, 14), 28)

    wb.save(output_file)
    print(f"\nProceso finalizado con éxito. Archivo generado: '{output_file}'")


if __name__ == "__main__":
    procesar_calidad_aire_diaria_nom2023(
        input_file="BDPIN_Marzo_2024.xlsx",
        sheet_name="Data"
    )

Cargando archivo: BDPIN_Marzo_2024.xlsx (Hoja: 'Data')...

Proceso finalizado con éxito. Archivo generado: 'Calculos_Diarios_Aire_y_Salud.xlsx'
